# EDA 0 — Checkpoint B: Conceptual integrity (B09–B17)

**Technical evidence, not business KPIs.** This notebook examines the three unchanged CSV extracts and retains the distinctions among payment **attempt**, completed **assignment**, and **host**. No transaction amounts are available, so a populated `captured_at` does not prove full financial settlement. No data is dropped, repaired, deduplicated, or imputed.

**Execution:** Python 3.13; from the repository root or this notebook directory, place the original CSVs in `data/raw/`, choose the local `.venv` kernel, then Restart Kernel and Run All Cells. Reported results are specific to SHA-256 inputs displayed below. Prior Checkpoint A is a reference, not unquestioned proof of conceptual validity. Steps B09–B17 are self-contained; testable reused technical functions live in `src/foundation_b.py`.

**Decision boundary:** A question is answered only about directly observable fields. A preliminary technical flag does **not** define `Observed Payment Capture Rate`, payment recovery, or complete settlement. Detailed event chronology belongs to C; detailed method transitions and retries belong to D.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "data" / "raw").is_dir():
        root = candidate
        break
else:
    raise FileNotFoundError("Run inside the repository; data/raw and pyproject.toml are required")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.foundation_a import load_raw, assert_conserved
from src.foundation_b import status_capture_matrix, project_capture_evidence, first_completion_flag_audit
frames, provenance = load_raw(root)
e, a, p = frames['exposures'], frames['assignments'], frames['payments']
assert (len(e),len(a),len(p)) == (14708,279847,306957), 'Extract differs from Checkpoint A: examine changes before applying prior totals'
display(provenance)
print('Reference rows (exposures, assignments, payment attempts):', len(e), len(a), len(p))

Reference rows (exposures, assignments, payment attempts): 14708 279847 306957


,dataset,file,bytes,sha256,columns_unexpected
0,exposures,experiment_exposures.csv,481724,9cce67cd8d90d292c0703416b3c28c69783c56d10c2f997bf614b2bc84445327,
1,assignments,assignments.csv,16411731,c66a6d5cb89b62bc3542ee9184e75c8ded41589e91dac9d726639acad0db0ff2,
2,payments,payment_logs.csv,24857672,ab6bf97f2ca216504058b5de1deb2f80641d783e8e23651d3efc2d595675879a,


## B09 — Nulls and co-occurring missing values

**Procedure:** count null and non-null source fields independently for each dataset, then inspect capture-status-manual combinations. **Validation:** each column's null/non-null counts reconcile exactly to its table size. Empty-string parsing follows Checkpoint A; missingness does not imply an unpaid *project*.

In [2]:
null_rows = []
for name, df in frames.items():
    for col in df.columns:
        missing = int(df[col].isna().sum())
        present = int(df[col].notna().sum())
        assert_conserved(len(df), missing, present)
        assert missing == int(len(df) - df[col].count()) # independent cross-check
        null_rows.append({'dataset':name, 'field':col, 'null_rows':missing, 'not_null_rows':present,
                          'dataset_rows':len(df), 'null_pct':round(100*missing/len(df),5)})
nullity = pd.DataFrame(null_rows)
display(nullity)
missing_combinations = (p.assign(capture_present=p.captured_at.notna())
  .groupby(['payment_status','is_manual_attempt','capture_present'],dropna=False)
  .size().rename('attempts').reset_index().sort_values('attempts',ascending=False))
assert int(missing_combinations.attempts.sum()) == len(p)
display(missing_combinations)

,dataset,field,null_rows,not_null_rows,dataset_rows,null_pct
0,exposures,host_id,0,14708,14708,0.00000
1,exposures,experiment_group,0,14708,14708,0.00000
2,exposures,first_exposed_at,0,14708,14708,0.00000
3,assignments,assignment_id,0,279847,279847,0.00000
4,assignments,host_id,0,279847,279847,0.00000
5,assignments,created_at,0,279847,279847,0.00000
6,assignments,completed_at,0,279847,279847,0.00000
7,assignments,is_first_completed_project,0,279847,279847,0.00000
8,payments,payment_log_id,0,306957,306957,0.00000
9,payments,assignment_id,0,306957,306957,0.00000


,payment_status,is_manual_attempt,capture_present,attempts
8,succeeded,false,True,272445
0,declined,false,False,12820
11,voided,false,False,9392
10,succeeded,true,True,4763
2,declined,true,False,4371
12,voided,false,True,1309
3,failed,false,False,849
4,failed,false,True,523
7,succeeded,false,False,217
6,failed,true,True,137


**B09 technical status: Answered for the supplied source fields.** The `captured_at` null at attempt grain cannot establish a project's eventual state because projects can have multiple attempts. Patterns involving missing capture are investigated in B12 and B16.

## B10 — Declared versus observed categorical domains

**Procedure:** display every distinct observed categorical value (including nulls), its frequency, the documented valid domain, and unexpected values. Preserve literal source strings; do not normalize case, whitespace or booleans. **Validation:** category frequencies plus missing values equal the field's source row count.

In [3]:
allowed = {'experiment_group': {'control','test'}, 'payment_method': {'card','bank_account','credits'},
           'payment_status': {'succeeded','declined','failed','voided'},
           'is_manual_attempt': {'true','false'}, 'is_first_completed_project': {'true','false'}}
source = {'experiment_group':e, 'payment_method':p, 'payment_status':p,
          'is_manual_attempt':p, 'is_first_completed_project':a}
rows=[]
for col, domain in allowed.items():
    df=source[col]
    values=df[col].value_counts(dropna=False)
    assert int(values.sum()) == len(df)
    for value, count in values.items():
        rows.append({'field':col,'source_value':str(value),'attempt_or_project_or_host_count':int(count),
                     'expected_domain':', '.join(sorted(domain)),
                     'unexpected': bool(pd.isna(value) or value not in domain)})
domains=pd.DataFrame(rows)
display(domains)
assert int(domains.loc[domains.field=='payment_status','attempt_or_project_or_host_count'].sum()) == len(p)
print('Unexpected domain values (rows of value counts):',int(domains.unexpected.sum()))
# Timestamp type/format check is diagnostic only; source string fields remain unchanged.
format_diagnostics=[]
for dataset,frame,cols in [('exposures',e,['first_exposed_at']),
                           ('assignments',a,['created_at','completed_at']),
                           ('payments',p,['created_at','captured_at'])]:
    for col in cols:
        parsed=pd.to_datetime(frame[col],format='mixed',utc=True,errors='coerce')
        invalid=int((frame[col].notna() & parsed.isna()).sum())
        format_diagnostics.append({'dataset':dataset,'field':col,
          'source_dtype':str(frame[col].dtype),'present_values':int(frame[col].notna().sum()),
          'unparseable_present_values':invalid,'parsed_timezone':'UTC'})
formats=pd.DataFrame(format_diagnostics)
display(formats)


Unexpected domain values (rows of value counts): 0


,field,source_value,attempt_or_project_or_host_count,expected_domain,unexpected
0,experiment_group,control,7377,"control, test",False
1,experiment_group,test,7331,"control, test",False
2,payment_method,card,207031,"bank_account, card, credits",False
3,payment_method,bank_account,99293,"bank_account, card, credits",False
4,payment_method,credits,633,"bank_account, card, credits",False
5,payment_status,succeeded,277447,"declined, failed, succeeded, voided",False
6,payment_status,declined,17231,"declined, failed, succeeded, voided",False
7,payment_status,voided,10769,"declined, failed, succeeded, voided",False
8,payment_status,failed,1510,"declined, failed, succeeded, voided",False
9,is_manual_attempt,false,297595,"false, true",False


,dataset,field,source_dtype,present_values,unparseable_present_values,parsed_timezone
0,exposures,first_exposed_at,string,14708,0,UTC
1,assignments,created_at,string,279847,0,UTC
2,assignments,completed_at,string,279847,0,UTC
3,payments,created_at,string,306957,0,UTC
4,payments,captured_at,string,279285,0,UTC


**B10 technical status: Answered for recorded categories.** Data types follow the source-preserving string import from A; these checks do not establish semantics not recorded by fields.

## B11 — Multiple success statuses and capture records

**Procedure:** build one diagnostic record for each original assignment, containing both distinct status-success rows and non-null capture rows. Explicitly distinguish multiple captured *rows*, multiple *distinct timestamps*, and multiple *methods among captured attempts*. **Validation:** source payment attempts and populated capture timestamps reconcile after aggregation; no join multiplies rows.

These occurrences cannot be interpreted as duplicate charges, complementary payments or full settlement without financial amounts.

In [4]:
project = project_capture_evidence(a,p)
assert len(project) == len(a) and project.assignment_id.nunique() == len(a)
assert project.attempt_rows.sum() == len(p)
assert project.captured_field_rows.sum() == p.captured_at.notna().sum()
assert_conserved(len(a), int(project.has_logs.sum()), int((~project.has_logs).sum()))
summary_b11 = pd.DataFrame([
    {'condition':'2+ succeeded-status attempts','projects':int(project.success_status_rows.ge(2).sum())},
    {'condition':'2+ populated captured_at rows','projects':int(project.multiple_capture_rows.sum())},
    {'condition':'2+ distinct captured_at timestamp strings','projects':int(project.multiple_distinct_capture_times.sum())},
    {'condition':'captures recorded by 2+ methods','projects':int(project.capture_multiple_methods.sum())},
])
display(summary_b11)
display(project.loc[project.multiple_capture_rows,
    ['assignment_id','attempt_rows','success_status_rows','captured_field_rows','distinct_capture_timestamps','distinct_capture_methods']].head(8))
print('Captured field rows, verified against source:',int(project.captured_field_rows.sum()))

Captured field rows, verified against source: 279285


,condition,projects
0,2+ succeeded-status attempts,154
1,2+ populated captured_at rows,652
2,2+ distinct captured_at timestamp strings,652
3,captures recorded by 2+ methods,200


,assignment_id,attempt_rows,success_status_rows,captured_field_rows,distinct_capture_timestamps,distinct_capture_methods
195,196,2,1,2,2,1
702,703,2,1,2,2,1
959,960,2,1,2,2,2
1532,1533,3,2,2,2,1
4385,4386,2,2,2,2,2
5067,5068,2,1,2,2,2
6605,6606,5,1,3,3,1
6646,6647,5,1,2,2,2


**B11 technical status: Answered for attempts and captured fields.** The counts characterize distinct log records, not independent financial movements. More detailed journey mechanics belong to D.

## B12 — Full status × captured_at cross-tab (priority control)

**Procedure:** cross `payment_status` with presence of `captured_at`. Discordance is defined **only for this technical diagnostic** as `succeeded` without timestamp OR non-`succeeded` with timestamp. This is a field-semantic comparison, **not** a correction rule.

**Validation:** row and column margins, source-level boolean masks, group-specific direct counts and independently calculated raw count difference. Inspect impact at assignment grain and by payment method.

In [5]:
matrix = status_capture_matrix(p)
display(matrix)
totals = matrix[['captured_at_missing','captured_at_present']].sum()
assert int(matrix.all_attempts.sum()) == len(p)
assert int(totals.captured_at_missing) == int(p.captured_at.isna().sum())
assert int(totals.captured_at_present) == int(p.captured_at.notna().sum())
assert int(matrix.loc['succeeded','all_attempts']) == int(p.payment_status.eq('succeeded').sum())

success_no_capture = p.payment_status.eq('succeeded') & p.captured_at.isna()
non_success_with_capture = p.payment_status.ne('succeeded') & p.captured_at.notna()
discord = success_no_capture | non_success_with_capture
assert int(discord.sum()) == int(success_no_capture.sum()+non_success_with_capture.sum())
raw_success_count=int(p.payment_status.eq('succeeded').sum())
raw_capture_count=int(p.captured_at.notna().sum())
assert raw_capture_count - raw_success_count == int(non_success_with_capture.sum()) - int(success_no_capture.sum())
print({'success_without_timestamp_attempts':int(success_no_capture.sum()),
       'non_success_with_timestamp_attempts':int(non_success_with_capture.sum()),
       'all_discordant_attempts':int(discord.sum()),
       'raw_capture_minus_success_difference':raw_capture_count-raw_success_count,
       'distinct_affected_assignments':p.loc[discord,'assignment_id'].nunique()})

b12_methods=(p.assign(discordant=discord,success_no_timestamp=success_no_capture,
                      non_success_with_timestamp=non_success_with_capture)
             .groupby('payment_method',dropna=False)
             .agg(attempts=('payment_log_id','size'),discordant_attempts=('discordant','sum'),
                  success_without_timestamp=('success_no_timestamp','sum'),
                  other_status_with_timestamp=('non_success_with_timestamp','sum')))
assert int(b12_methods.attempts.sum()) == len(p)
assert int(b12_methods.discordant_attempts.sum()) == int(discord.sum())
display(b12_methods)

concordant=(p.payment_status.eq('succeeded') & p.captured_at.notna())
ids_discord=set(p.loc[discord,'assignment_id'])
ids_concord=set(p.loc[concordant,'assignment_id'])
print('Affected projects also containing another concordant attempt:',len(ids_discord & ids_concord))
display(p.loc[discord,['payment_log_id','assignment_id','payment_method','payment_status','captured_at']].head(8))

{'success_without_timestamp_attempts': 239, 'non_success_with_timestamp_attempts': 2077, 'all_discordant_attempts': 2316, 'raw_capture_minus_success_difference': 1838, 'distinct_affected_assignments': 2075}
Affected projects also containing another concordant attempt: 595


,captured_at_missing,captured_at_present,all_attempts
payment_status,,,
declined,17191,40,17231
failed,850,660,1510
succeeded,239,277208,277447
voided,9392,1377,10769


,attempts,discordant_attempts,success_without_timestamp,other_status_with_timestamp
payment_method,,,,
bank_account,99293,976,0,976
card,207031,1326,239,1087
credits,633,14,0,14


,payment_log_id,assignment_id,payment_method,payment_status,captured_at
201,202,54592,bank_account,voided,2025-07-19 00:01:17
206,207,55254,bank_account,voided,2025-07-18 00:04:17
275,276,28270,bank_account,voided,2025-07-18 00:04:51
294,295,53256,bank_account,failed,2025-07-19 00:01:48
318,319,34089,bank_account,voided,2025-07-18 00:05:01
400,401,22884,card,voided,2025-07-19 00:02:46
453,454,56596,card,voided,2025-07-19 00:03:01
504,505,24410,bank_account,voided,2025-07-18 13:45:00


**B12 technical status: Answered for raw field disagreement.** A marginal difference does not equal total disagreement. Neither field is selected as the business source of truth; its consequences are examined in B16. The origin of these discordant logs is unknown.

## B13 — Historical first-completion flag

**Procedure:** count positive flags per host, contrast flagged project completion with the earliest *observed* completion in the extract, and disclose timestamp ties. **Validation:** flagged projects reconcile to source, and host project counts reconcile to assignments. The extract begins in July 2025; missing earlier history or platform registration dates cannot be inferred.

In [6]:
flag = first_completion_flag_audit(a)
print({'projects_with_true_flag':int(a.is_first_completed_project.eq('true').sum()),
       'hosts_with_flag':int(flag.positive_flags.gt(0).sum()),
       'hosts_with_multiple_true_flags':int(flag.positive_flags.gt(1).sum()),
       'flagged_projects_later_than_observed_min':int(flag.flagged_later_than_observed_earliest.sum()),
       'hosts_tied_at_earliest_observed_completion':int(flag.earliest_observed_ties.gt(1).sum()),
       'flagged_projects_with_invalid_completed_at':int(flag.flagged_timestamp_invalid.sum())})
assert int(flag.observed_projects.sum()) == len(a)
assert int(flag.positive_flags.sum()) == int(a.is_first_completed_project.eq('true').sum())
display(flag[['observed_projects','positive_flags','earliest_observed_ties']].describe().round(2))
display(flag.loc[(flag.positive_flags.gt(1) | flag.flagged_later_than_observed_earliest.gt(0))].head(8))

{'projects_with_true_flag': 2755, 'hosts_with_flag': 2755, 'hosts_with_multiple_true_flags': 0, 'flagged_projects_later_than_observed_min': 0, 'hosts_tied_at_earliest_observed_completion': 7, 'flagged_projects_with_invalid_completed_at': 0}


,observed_projects,positive_flags,earliest_observed_ties
count,14066.00,14066.0,14066.00
mean,19.90,0.2,1.00
std,43.01,0.4,0.04
min,1.00,0.0,1.00
25%,4.00,0.0,1.00
50%,11.00,0.0,1.00
75%,22.00,0.0,1.00
max,1741.00,1.0,5.00


,host_id,observed_projects,positive_flags,flagged_later_than_observed_earliest,flagged_timestamp_invalid,earliest_observed_ties


**B13 technical status: Partially answerable.** Chronology and repeated flags are checkable within the extract only; the complete historical first project cannot be independently authenticated.

## B14 — Exact duplicate rows, repeated IDs, and similar content with new IDs

**Procedure:** distinguish duplicate primary identifiers, identical full rows and exact matches on all fields *except* `payment_log_id`. Also show a broader same-project/status/method similarity for context. **Validation:** duplicates are counted as excess rows and grouped collision frequencies; every source row remains in place.

In [7]:
nonid=[col for col in p.columns if col != 'payment_log_id']
full_excess=int(p.duplicated(keep='first').sum())
key_excess=int(p.payment_log_id.duplicated(keep='first').sum())
content_excess=int(p.duplicated(subset=nonid,keep='first').sum())
content_members=int(p.duplicated(subset=nonid,keep=False).sum())
content_sizes=p.groupby(nonid,dropna=False).size().rename('rows').reset_index()
same_content_groups=int(content_sizes.rows.gt(1).sum())
assert int(content_sizes.rows.sum())==len(p)
assert content_excess == int((content_sizes.rows-1).clip(lower=0).sum())
print({'repeated_payment_log_id_excess_rows':key_excess,'identical_full_row_excess':full_excess,
       'identical_attributes_except_ID_excess':content_excess,'rows_in_identical_attribute_groups':content_members,
       'groups_with_repeated_non_ID_attributes':same_content_groups})
display(content_sizes.loc[content_sizes.rows.gt(1)].sort_values('rows',ascending=False).head(8))
broader=['assignment_id','payment_method','payment_status']
print('Rows in same-project/method/status groups of size >=2:',
      int(p.duplicated(subset=broader,keep=False).sum()), '(similarity only; not a duplicate definition)')

{'repeated_payment_log_id_excess_rows': 0, 'identical_full_row_excess': 0, 'identical_attributes_except_ID_excess': 0, 'rows_in_identical_attribute_groups': 0, 'groups_with_repeated_non_ID_attributes': 0}
Rows in same-project/method/status groups of size >=2: 17986 (similarity only; not a duplicate definition)


,assignment_id,host_id,payment_method,payment_status,is_manual_attempt,created_at,captured_at,rows


**B14 technical status: Answered for exact/similar source content.** Distinct attempt IDs with identical other fields can be legitimate separate attempts and are preserved.

## B15 — Observable status combinations; limits of sequence ordering

**Procedure:** characterize unordered combinations of statuses by assignment, and the scope of tied payment-record `created_at` timestamps. Show only technical examples where complete and unique `created_at` values permit record ordering. **Validation:** status-combination categories plus no-log projects reconcile to all assignments; no artificial tie-breaking is used.

The *record creation sequence* is **not** evidence of the actual timing of charge failure or financial recovery. Full journey analysis belongs to D.

In [8]:
# Vectorized presence table avoids Python callbacks for each of ~280k projects.
status_presence=(p.assign(_status=p.payment_status.fillna('<MISSING>'))
  .groupby(['assignment_id','_status']).size().unstack(fill_value=0).gt(0))
status_signature=pd.Series('',index=status_presence.index,dtype='string')
for status in sorted(status_presence.columns):
    status_signature=status_signature+pd.Series(np.where(status_presence[status],status+'|',''),index=status_presence.index)
combo_counts=status_signature.value_counts().rename('projects').reset_index()
combo_counts.columns=['observed_status_set','projects']
without_logs=int((~project.has_logs).sum())
assert_conserved(len(a),int(combo_counts.projects.sum()),without_logs)
display(combo_counts.head(15))

seen_success=status_presence.get('succeeded',pd.Series(False,index=status_presence.index))
seen_failed=(status_presence.get('declined',pd.Series(False,index=status_presence.index)) |
             status_presence.get('failed',pd.Series(False,index=status_presence.index)))
seen_void=status_presence.get('voided',pd.Series(False,index=status_presence.index))
print({'projects_with_success_and_failure_status':int((seen_success&seen_failed).sum()),
       'projects_with_success_and_voided_status':int((seen_success&seen_void).sum()),
       'projects_with_logs_but_no_success_status':int((~seen_success).sum()),
       'projects_without_logs':without_logs})

parsed_created=pd.to_datetime(p.created_at,format='mixed',utc=True,errors='coerce')
invalid_created=int(parsed_created.isna().sum())
# A timestamp tie means observed row order is not uniquely determined by created_at.
tied_project_count=int(p.groupby(['assignment_id','created_at'],dropna=False).size().loc[lambda s:s.gt(1)].reset_index()['assignment_id'].nunique())
print({'invalid_payment_created_at':invalid_created,
       'projects_with_equal_payment_log_created_at_among_2plus_attempts':tied_project_count})
# Examples strictly ordered by distinct created_at, without representing it as operation chronology.
non_tied_ids=p.groupby('assignment_id').size().loc[lambda s:s.gt(1)].index.difference(
    p.groupby(['assignment_id','created_at'],dropna=False).size().loc[lambda s:s.gt(1)].reset_index()['assignment_id'].unique())
selected_ids=list(non_tied_ids[:4])
display(p.loc[p.assignment_id.isin(selected_ids),['assignment_id','payment_log_id','created_at','payment_status']].sort_values(['assignment_id','created_at']).head(15))

{'projects_with_success_and_failure_status': 7125, 'projects_with_success_and_voided_status': 6662, 'projects_with_logs_but_no_success_status': 2317, 'projects_without_logs': 239}
{'invalid_payment_created_at': 0, 'projects_with_equal_payment_log_created_at_among_2plus_attempts': 1}


,observed_status_set,projects
0,succeeded|,264199
1,succeeded|voided|,5967
2,declined|succeeded|,5290
3,voided|,1417
4,failed|succeeded|,938
5,declined|,722
6,declined|succeeded|voided|,533
7,declined|failed|succeeded|,202
8,declined|failed|succeeded|voided|,135
9,declined|failed|,76


,assignment_id,payment_log_id,created_at,payment_status
59532,100020,59533,2025-08-07 07:26:38,voided
75759,100020,75760,2025-08-13 00:09:24,succeeded
44072,100026,44073,2025-08-01 12:37:31,declined
44448,100026,44449,2025-08-01 13:45:26,succeeded
27580,10005,27581,2025-07-27 09:00:28,failed
32818,10005,32819,2025-07-28 16:05:46,succeeded
77059,100055,77060,2025-08-13 09:30:32,voided
98043,100055,98044,2025-08-22 00:07:50,succeeded


**B15 technical status: Partially answerable.** Status combinations and record ordering can be observed; exact sequence of underlying financial operations or retries cannot be verified solely from creation timestamps.

## B16 — Assignment-level capture evidence, without a business capture-rate definition

**Procedure:** preserve *all* completed projects, and display overlapping evidence: any success status, any populated capture timestamp, at least one concordant row, at least one discordant row, no log, multiple captures and first valid capture timestamp. Build a separate **exclusive diagnostic review-priority partition** for reconciling the project universe. This partition is *not* a success/failure label or approved KPI rule.

**Validation:** source attempt/capture counts, project total and both independent membership checks reconcile to original CSVs. The presence of a first captured timestamp is only an observable event, **not** proof of fully paid amounts. No capture recorded is not financial loss.

In [9]:
evidence_columns=['has_logs','any_success_status','any_capture_field','any_concordant','any_discordant',
                  'multiple_capture_rows','multiple_distinct_capture_times','capture_multiple_methods',
                  'first_valid_capture_identifiable','potentially_ambiguous']
evidence_b16=pd.DataFrame({'diagnostic_evidence':evidence_columns,
                           'assignments':[int(project[c].sum()) for c in evidence_columns]})
display(evidence_b16)
assert project.has_logs.sum() == a.assignment_id.isin(p.assignment_id).sum()
assert project.any_capture_field.sum() == p.loc[p.captured_at.notna(),'assignment_id'].nunique()
assert project.any_success_status.sum() == p.loc[p.payment_status.eq('succeeded'),'assignment_id'].nunique()
assert project.any_discordant.sum() == p.loc[discord,'assignment_id'].nunique()
assert int(project.first_valid_capture_identifiable.sum()) <= int(project.any_capture_field.sum())

# Exclusive categories are only for diagnostic reconciliation; precedence is explicitly declared.
conditions=[~project.has_logs,project.any_discordant,project.multiple_capture_rows,
            project.any_concordant,project.any_capture_field,project.any_success_status]
labels=['no_payment_log','at_least_one_discordant_attempt','multiple_capture_rows_no_discordance',
        'concordant_capture_no_above_flags','capture_without_status_success_no_above_flags',
        'success_status_without_capture_no_above_flags']
project['diagnostic_partition']=np.select(conditions,labels,default='logs_no_success_no_capture')
partition=project.diagnostic_partition.value_counts().rename_axis('review_category').reset_index(name='assignments')
assert int(partition.assignments.sum())==len(a)
assert int(partition.loc[partition.review_category=='no_payment_log','assignments'].sum())==239
display(partition)
print('Observed first valid capture timestamp:',int(project.first_valid_capture_identifiable.sum()),
      '| Captured fields without parseable first capture:',
      int((project.any_capture_field & ~project.first_valid_capture_identifiable).sum()))

Observed first valid capture timestamp: 278559 | Captured fields without parseable first capture: 0


,diagnostic_evidence,assignments
0,has_logs,279608
1,any_success_status,277291
2,any_capture_field,278559
3,any_concordant,277115
4,any_discordant,2075
5,multiple_capture_rows,652
6,multiple_distinct_capture_times,652
7,capture_multiple_methods,200
8,first_valid_capture_identifiable,278559
9,potentially_ambiguous,2402


,review_category,assignments
0,concordant_capture_no_above_flags,276432
1,at_least_one_discordant_attempt,2075
2,logs_no_success_no_capture,1013
3,no_payment_log,239
4,multiple_capture_rows_no_discordance,88


### V8 independent cross-check — standard-library CSV scan

Read the original payment CSV a second time with Python's `csv.DictReader`, independently of Pandas crosstabs and joins. Reconcile the four status × capture cells, affected project IDs, per-project populated capture counts and any status-success presence. This control does **not** resolve the conceptual mismatch; it detects implementation errors.

In [10]:
import csv
from collections import Counter
raw_status_capture=Counter()
raw_captured_projects=set()
raw_success_projects=set()
raw_discordant_projects=set()
raw_capture_counts=Counter()
raw_rows=0
with (root/'data'/'raw'/'payment_logs.csv').open(encoding='utf-8-sig',newline='') as f:
    for row in csv.DictReader(f):
        raw_rows+=1
        status=row['payment_status']
        has_timestamp=bool(row['captured_at'])
        assignment=row['assignment_id']
        raw_status_capture[(status,has_timestamp)]+=1
        if status=='succeeded':raw_success_projects.add(assignment)
        if has_timestamp:
            raw_captured_projects.add(assignment)
            raw_capture_counts[assignment]+=1
        if (status=='succeeded') != has_timestamp:
            raw_discordant_projects.add(assignment)
assert raw_rows==len(p)
for status in matrix.index:
    assert raw_status_capture[(status,False)]==int(matrix.loc[status,'captured_at_missing'])
    assert raw_status_capture[(status,True)]==int(matrix.loc[status,'captured_at_present'])
assert len(raw_captured_projects)==int(project.any_capture_field.sum())
assert len(raw_success_projects)==int(project.any_success_status.sum())
assert len(raw_discordant_projects)==int(project.any_discordant.sum())
assert sum(v>=2 for v in raw_capture_counts.values())==int(project.multiple_capture_rows.sum())
print({'independent_csv_attempts':raw_rows,'independent_captured_projects':len(raw_captured_projects),
       'independent_success_status_projects':len(raw_success_projects),
       'independent_discordant_projects':len(raw_discordant_projects),
       'independent_multiple_captured_projects':sum(v>=2 for v in raw_capture_counts.values()),
       'all_independent_reconciliations':'PASS'})

{'independent_csv_attempts': 306957, 'independent_captured_projects': 278559, 'independent_success_status_projects': 277291, 'independent_discordant_projects': 2075, 'independent_multiple_captured_projects': 652, 'all_independent_reconciliations': 'PASS'}


**B16 technical status: Partially answerable for financial interpretation, answered for recorded evidence.** The Boolean indicators overlap; the exclusive partition follows the stated priority only and must never be treated as payment outcome. Even a concordant timestamp cannot prove the due amount was fully settled. All 239 no-log projects are retained.

## B17 — Which operational events are identifiable from supplied fields?

**Procedure:** compare available fields to each proposed real-world operation. This is a matrix of *direct evidence*, *indirect signals*, *missing information* and *technical limitation*. A record timestamp does not identify the underlying operation unless the source explicitly says so.

In [11]:
event_matrix=pd.DataFrame([
    ('Pre-authorization','payment created_at; group; card method','No explicit event type','Early record creation may coincide','authorization action ID, type, status, hold start/end','Cannot classify a row as an authorization'),
    ('Re-authorization','payment created_at; group; card method','No explicit reauth event','Multiple records may exist','authorization attempt type, parent hold ID','Cannot distinguish reauth from other records'),
    ('Charge / capture','payment_status; captured_at','Captured timestamp populated and status recorded','A timestamp is evidence of a reported capture','amount, settlement and transfer ledger','Cannot verify full amount paid'),
    ('Automatic attempt','is_manual_attempt','Recorded false manual flag','System triggering is suggested by flag','scheduler/retry reason and event timestamp','False is not a full event log'),
    ('Manual attempt','is_manual_attempt','Recorded true manual flag','May indicate support recovery','CS action and contact log','Does not capture all CS outreach'),
    ('Payment failure','payment_status','Declined/failed status of an attempt','Record created_at may precede failure','timestamp of actual failure','Cannot date failure from created_at alone'),
    ('Recovery after failure','status; captured_at; project ID','No recovery event type','Failure-status and capture evidence may coexist','failure and retry event times, reason','Causality and recovery-start time unknown'),
    ('Payment-method change','payment_method; project ID','Methods for attempts are directly recorded','Different methods may appear on same project','host choice and method-change event','Cannot attribute intention to host'),
    ('Full financial settlement','captured_at; payment_status','No complete amount evidence','Capture timestamp is a partial signal','amount due, captured amounts, reversals','Full settlement cannot be established')
],columns=['event','available_fields','direct_observation','indirect_signal','missing_fields','technical_boundary'])
display(event_matrix)
assert len(event_matrix)==9

,event,available_fields,direct_observation,indirect_signal,missing_fields,technical_boundary
0,Pre-authorization,payment created_at; group; card method,No explicit event type,Early record creation may coincide,"authorization action ID, type, status, hold start/end",Cannot classify a row as an authorization
1,Re-authorization,payment created_at; group; card method,No explicit reauth event,Multiple records may exist,"authorization attempt type, parent hold ID",Cannot distinguish reauth from other records
2,Charge / capture,payment_status; captured_at,Captured timestamp populated and status recorded,A timestamp is evidence of a reported capture,"amount, settlement and transfer ledger",Cannot verify full amount paid
3,Automatic attempt,is_manual_attempt,Recorded false manual flag,System triggering is suggested by flag,scheduler/retry reason and event timestamp,False is not a full event log
4,Manual attempt,is_manual_attempt,Recorded true manual flag,May indicate support recovery,CS action and contact log,Does not capture all CS outreach
5,Payment failure,payment_status,Declined/failed status of an attempt,Record created_at may precede failure,timestamp of actual failure,Cannot date failure from created_at alone
6,Recovery after failure,status; captured_at; project ID,No recovery event type,Failure-status and capture evidence may coexist,"failure and retry event times, reason",Causality and recovery-start time unknown
7,Payment-method change,payment_method; project ID,Methods for attempts are directly recorded,Different methods may appear on same project,host choice and method-change event,Cannot attribute intention to host
8,Full financial settlement,captured_at; payment_status,No complete amount evidence,Capture timestamp is a partial signal,"amount due, captured amounts, reversals",Full settlement cannot be established


**B17 technical status: Partially answerable.** The data documents attempt-level statuses, some capture timestamps and an attempt-manual flag, but not complete pre-authorization, reauthorization, recovery, charge amounts or customer-support histories. Avoid imposing undocumented event codes.

---

## Checkpoint summary and review requirements

- **B09–B17:** evidence tables, validations, limitations and technical-status statements appear directly in sections above. This is not a metrics catalog.
- **Primary unresolved decision:** how to use discordant `payment_status` versus `captured_at` for future business metrics; neither field is designated the source of truth.
- **Dependencies:** C will investigate temporal coverage and incident timing; D will investigate full attempt and method journeys; Fernando and Jurandir approve eventual business definitions.
- **V7:** the notebook must be run from a clean Jupyter kernel; do not mark that control passed unless the complete run is independently confirmed. Synthetic pytest tests validate the helper behavior but do not establish Jupyter reproducibility.
- **No commits or pull requests** for this checkpoint without approval; do not advance to C automatically.